# Model Training and Validation

This notebook presents the training procedure used for all ResNet-18 models evaluated in this study. The experiments include both standard training and adversarial training, with and without data augmentation, in order to compare the effect of different training strategies on model performance and robustness.

## Setup

This section imports the required libraries, defines the computational device, and applies a consistent plotting style for all visualizations generated during training.

In [ ]:
from torchvision.transforms import v2 
import torchvision
import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt
import numpy as np 

from thesis_adversarial import resnet18, PGDAttack

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams["font.family"] = "serif"

## Training Configuration
The following parameters define the optimization settings, learning rate schedule, and adversarial attack parameters used during training.

In [ ]:
batch_size = 128
learning_rate = 0.1
epochs = 100 
weight_decay = 5e-4
momentum = 0.9

milestones = [30, 60]
gamma = 0.1

epsilon = 0.03
steps = 10
alpha = 0.006

classes = ('airlane','car','bird','cat','deer','dog','frog','horse','ship','truck')

## Training Utilities

The helper function degined below implement the training loops and the plotting used throughout the norebook.

### Standard Training

This training routine performs standard supervised learning on clean input images. At the end of each epoch, the model is evaluated on the validation set and the best performing checkpoint is stored based on validation accuracy.

In [ ]:
def standard_train(
    model, 
    trainloader, 
    valloader,
    optimizer, 
    scheduler, 
    loss_fn,
    epochs
    ):
    
    results ={
        'train_loss' : [],
        'train_accuracy': [],
        'validation_loss': [],
        'validation_accuracy': []
    }
    
    best_val_acc = -1.0
    best_state_dict = None
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0
        
        for i, data in enumerate(trainloader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
    
            optimizer.zero_grad()
    
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(trainloader.dataset)
        train_acc = correct / total
        results['train_loss'].append(train_loss)
        results['train_accuracy'].append(train_acc)
        
        model.eval()
        total, correct = 0, 0
        running_loss = 0
        
        with torch.no_grad():
            for i, data in enumerate(valloader, 0):
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)
                
                running_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                
        val_acc = correct / total
        val_loss = running_loss / len(valloader.dataset)
        results['validation_loss'].append(val_loss)
        results['validation_accuracy'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        
        scheduler.step()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f} | "
            f"Val Clean Loss: {val_loss:.4f} | Val Clean Acc: {val_acc:.2f} | "
        )
            
    print('Finished Training')
    return results, best_state_dict, best_val_acc

### Adversarial Training

This training routine performs adversarial training using adversarial examples generated by the specified attack function. During validation the model is evaluated on both clean and adversarial inputs. The best performing checkpoint is selected based on adversarial accuracy. 

In [ ]:
def adversarial_train(
    model, 
    trainloader, 
    valloader, 
    optimizer, 
    scheduler, 
    loss_fn, 
    epochs, 
    attack
    ):
    
    results ={
        'train_loss' : [],
        'train_accuracy': [],
        'validation_loss': [],
        'validation_accuracy': [],
        'validation_robust_accuracy': [],
        'validation_adversarial_loss': [],
        'pre_gen_validation_adversarial_loss': []
    }
    
    best_val_adv_acc = -1.0
    best_state_dict = None
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0
        pre_attacked_inputs = []
        pre_attacked_labels = []
        
        for inputs, labels in valloader:
            inputs, labels = inputs.to(device), labels.to(device)
            adv_inputs = attack.perturb(inputs, labels)
            
            pre_attacked_inputs.append(adv_inputs)
            pre_attacked_labels.append(labels)
            
        
        for inputs, labels,  in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
    
            optimizer.zero_grad()
            
            adv_inputs = attack.perturb(inputs, labels)
    
            outputs = model(adv_inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(trainloader.dataset)
        train_acc = correct / total
        results['train_loss'].append(train_loss)
        results['train_accuracy'].append(train_acc)
        
        model.eval()
        
        pre_adv_running_loss = 0.0 
        
        for i in range(len(pre_attacked_inputs)):
            with torch.no_grad():
                inputs, labels  = pre_attacked_inputs[i], pre_attacked_labels[i]
                
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)
            
            pre_adv_running_loss += loss.item() * inputs.size(0)
        
        total, clean_correct, adv_correct = 0, 0, 0
        clean_running_loss = 0.0
        adv_running_loss = 0.0 
        
        for inputs, labels  in valloader:
            inputs, labels = inputs.to(device), labels.to(device)
            adv_inputs = attack.perturb(inputs, labels)
        
            with torch.no_grad():
                clean_outputs = model(inputs)
                adv_outputs = model(adv_inputs)
                clean_loss = loss_fn(clean_outputs, labels)
                adv_loss = loss_fn(adv_outputs, labels)
                
            clean_running_loss += clean_loss.item() * inputs.size(0)
            adv_running_loss += adv_loss.item() * inputs.size(0)
            
            clean_correct += clean_outputs.argmax(1).eq(labels).sum().item()
            adv_correct += adv_outputs.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
        
        val_clean_acc = clean_correct / total
        val_robust_acc = adv_correct / total
        val_clean_loss = clean_running_loss / len(valloader.dataset)
        val_adv_loss = adv_running_loss / len(valloader.dataset)
        pre_val_adv_loss = pre_adv_running_loss /len(valloader.dataset)
        
        
        results['validation_accuracy'].append(val_clean_acc)
        results['validation_robust_accuracy'].append(val_robust_acc)
        results['validation_loss'].append(val_clean_loss)
        results['validation_adversarial_loss'].append(val_adv_loss)
        results['pre_gen_validation_adversarial_loss'].append(pre_val_adv_loss)
        
        if val_robust_acc > best_val_adv_acc:
            best_val_adv_acc = val_robust_acc
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        
        scheduler.step()
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f} | "
            f"Val Clean Loss: {val_clean_loss:.4f} | Val Clean Acc: {val_clean_acc:.2f} | "
            f"Val Adv Loss: {val_adv_loss:.4f} | Val Adv Acc: {val_robust_acc:.2f} | "
            f"Pre Optimize Generated Adv Loss {pre_val_adv_loss}"
        )

            
    print('Finished Training')
    return results, best_state_dict, best_val_adv_acc
    

In [ ]:
def plot_results(results, save=False, filename=None):
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))

    axs[0].plot(results['train_loss'], label='Train Adv Loss', color='lightskyblue')
    axs[0].plot(results['validation_loss'], label='Validation Clean Loss', color='orange')
    axs[0].set_xlabel('Epochs')
    axs[0].set_ylabel('Training Loss')
    axs[0].legend()

    axs[1].plot(results['train_accuracy'], label='Training Robust Acc', color='lightskyblue')
    axs[1].plot(results['validation_accuracy'], label='Validation Clean Acc', color='orange')
    axs[1].set_xlabel('Epochs')
    axs[1].set_ylabel('Accuracy')
    axs[1].legend()

    fig.subplots_adjust(wspace=0.25)

    if save:
        fig.savefig(f'../plots/plot_{filename}.png', dpi=300, bbox_inches='tight')

    plt.show()
    
    is_adv_run = results.get('validation_robust_accuracy') is not None and results.get('validation_adversarial_loss') is not None and results.get('pre_gen_validation_adversarial_loss') is not None
    
    if is_adv_run:
        fig, ax = plt.subplots()
        ax.plot(results['validation_adversarial_loss'], label='Validation Adv Loss', color='lightgreen')
        ax.plot(results['pre_gen_validation_adversarial_loss'], label='Pre Optimize Gen Adv loss', color='salmon')
        ax.legend()
    
        if save:
            fig.savefig(f'../figures/plot_adv_{filename}.pdf', dpi=300, bbox_inches='tight', format='pdf')
    
    plt.show()
        

In [ ]:
def show_training_examples(loader):
    
    dataiter = iter(loader )
    images, labels = next(dataiter)

    fig, axes = plt.subplots(4, 4, figsize=(8, 8))

    for i, ax in enumerate(axes.flat):
        ax.imshow(np.transpose(images[i].numpy(), (1, 2, 0)))
        ax.set_title(f'Class: {classes[labels[i]]}', fontsize=10)
        ax.set_axis_off()

    plt.subplots_adjust(wspace=0.25, hspace=0.1)
    plt.show()

# Training

## With augmentation

### Data Preparation

The CIFAR-10 training set is divided into a training split and a validation split. Different preprocessing piplines are defined in order to compare training with and without data augmentation

In [ ]:
transform_train = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomCrop(32, padding=4),
    v2.ToDtype(torch.float32, scale=True),
])

transform_val = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

generator = torch.Generator()
indices = torch.randperm(50_000, generator=generator)

train_indices = indices[:45_000]
val_indices = indices[45_000:]

train_dataset_full = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=True, transform=transform_train
)
val_dataset_full = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=False, transform=transform_val
)


trainset = torch.utils.data.Subset(train_dataset_full, train_indices)
valset = torch.utils.data.Subset(val_dataset_full, val_indices)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)


/Users/jonatanprepuk/Examensarbete/Code/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [8]:
# show_training_examples(trainloader)
# show_training_examples(adv_trainloader)
# show_training_examples(valloader)

### Model 1 - Standard training with data augmentation

<div style="display: flex;">

<div>

#### Training settings

| Setting | Value |
| :--- | :---: |
| Batch size | $128$ |
| Learning rate | $0.1$ |
| Epochs | $100$ |
| Weight decay | $5 \times 10^{-4}$ |
| Momentum | $0.9$ |
| Milestones | $30,\ 60$ |
| Gamma | $0.1$ |

</div>
</div>

In [ ]:
model = resnet18().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), learning_rate, momentum=momentum, 
                      weight_decay=weight_decay)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)


In [ ]:
results, best_state_dict, best_val_acc = standard_train(model, transform_train, valloader, 
                                                                  optimizer, scheduler, criterion, 
                                                                  epochs)

plot_results(results, save=True, filename='results_model_standard_training_with_aug.pdf')

torch.save(model.state_dict(), f'../models/model_standard_training_with_aug.pt.pt')
torch.save(best_state_dict, '../models/models/model_standard_training_with_aug_best.pt')

### Model 2 - Adversarial training with data augmentations

<div style="display: flex;">

<div>

#### Training settings

| Setting | Value |
| :--- | :---: |
| Batch size | $128$ |
| Learning rate | $0.1$ |
| Epochs | $100$ |
| Weight decay | $5 \times 10^{-4}$ |
| Momentum | $0.9$ |
| Milestones | $30,\ 60$ |
| Gamma | $0.1$ |

</div>

<div>

#### Attack settings

| Setting | Value |
| :--- | :---: |
| Type | $PGD$ |
| $\epsilon$ | $0.03$ |
| Steps | $10$ |
| $\alpha$ | $0.006$ |

</div>

</div>

In [9]:
model = resnet18().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), learning_rate, momentum=momentum, 
                      weight_decay=weight_decay)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)
attack = PGDAttack(model, epsilon, steps, alpha)

In [ ]:
results, best_state_dict, best_val_adv_acc = adversarial_train(model, transform_train, valloader, 
                                                                  optimizer, scheduler, criterion, 
                                                                  epochs, attack)

plot_results(results, save=True, filename='model_')

torch.save(model.state_dict(), f'../models/model_adversarial_training_best.pt')
torch.save(best_state_dict, '../models/model_adversarial_training_best.pt')

## Without augmentation

### Data

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

generator = torch.Generator()
indices = torch.randperm(50_000, generator=generator)

train_indices = indices[:45_000]
val_indices = indices[45_000:]

train_dataset_full = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=True, transform=transform
)
val_dataset_full = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=False, transform=transform
)

trainset = torch.utils.data.Subset(train_dataset_full, train_indices)
valset = torch.utils.data.Subset(val_dataset_full, val_indices)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)


In [ ]:
# show_training_examples(trainloader)
# show_training_examples(valloader)

### Model 3 - Standard training without augmentations


#### Training settings

| Setting | Value |
| :--- | :---: |
| Batch size | $128$ |
| Learning rate | $0.1$ |
| Epochs | $100$ |
| Weight decay | $5 \times 10^{-4}$ |
| Momentum | $0.9$ |
| Milestones | $30,\ 60$ |
| Gamma | $0.1$ |

In [ ]:
model = resnet18().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), learning_rate, momentum=momentum, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)

In [ ]:
results, best_state_dict, best_val_acc = standard_train(model, trainloader, valloader, 
                                                            optimizer, scheduler, criterion, 
                                                            epochs)

plot_results(results, save=True, filename='results_model_standard_training.pdf')

torch.save(model.state_dict(), f'../models/model_standard_training.pt')
torch.save(best_state_dict, '../models/model_standard_training_best.pt')